# Support Ticket Operations

**Level:** Intermediate

## Project description

A support team wants to understand response speed and backlog risk.
You will clean date columns, calculate response times, and summarize
tickets by priority and status.

## Skills practiced

- Parsing dates
- Calculating time differences
- Pivot tables
- Filtering operational risk


In [ ]:
import pandas as pd
import numpy as np
import secrets

practice_run_name = f"support-tickets-{secrets.token_hex(3)}"
print(f"Practice run: {practice_run_name}")

def make_ticket_data(row_count=35, seed=42):
    """Create a synthetic support ticket dataset for operations practice."""
    rng = np.random.default_rng(seed)
    created_at = (
        pd.Timestamp("2026-04-01 08:00")
        + pd.to_timedelta(rng.integers(0, 10 * 24 * 60, size=row_count), unit="m")
    )
    status = rng.choice(["Closed", "Open"], size=row_count, p=[0.7, 0.3])
    response_minutes = rng.integers(20, 8 * 60, size=row_count)

    tickets = pd.DataFrame({
        "ticket_id": range(2001, 2001 + row_count),
        "priority": rng.choice(["High", "Medium", "Low"], size=row_count, p=[0.25, 0.45, 0.30]),
        "status": status,
        "created_at": created_at,
    }).sort_values("created_at").reset_index(drop=True)

    tickets["first_response_at"] = tickets["created_at"] + pd.to_timedelta(response_minutes, unit="m")
    tickets.loc[tickets["status"] == "Open", "first_response_at"] = pd.NaT
    return tickets

tickets = make_ticket_data(row_count=35, seed=42)
tickets


## Step 1: Convert text dates into datetime columns


In [ ]:
work = tickets.copy()

work["created_at"] = pd.to_datetime(work["created_at"])
work["first_response_at"] = pd.to_datetime(work["first_response_at"])

work.info()


## Step 2: Calculate response time in hours


In [ ]:
work["response_hours"] = (
    work["first_response_at"] - work["created_at"]
).dt.total_seconds() / 3600

work


## Step 3: Summarize response time by priority


In [ ]:
priority_summary = (
    work
    .groupby("priority", as_index=False)
    .agg(
        ticket_count=("ticket_id", "count"),
        open_tickets=("status", lambda status: (status == "Open").sum()),
        average_response_hours=("response_hours", "mean"),
    )
    .sort_values("average_response_hours")
)

priority_summary


## Step 4: Create a backlog pivot table


In [ ]:
backlog = pd.pivot_table(
    work,
    index="priority",
    columns="status",
    values="ticket_id",
    aggfunc="count",
    fill_value=0,
)

backlog


## Step 5: Identify high-priority open tickets


In [ ]:
high_priority_open = work.loc[
    (work["priority"] == "High") & (work["status"] == "Open")
]

high_priority_open


## Final project notes

- Priority with slowest average response:
- Number of open high-priority tickets:
- Biggest operational risk:
- One staffing or process recommendation:


## Practice extension: Generate your own dataset

Change `row_count` and `seed` to simulate different ticket queues. Set
`seed=None` for a randomized dataset each run. Try making more rows and
compare whether the backlog risk changes.


In [ ]:
my_tickets = make_ticket_data(row_count=80, seed=99)
my_tickets.head()
